# Fault Tree Analysis (FTA) and Dependent Failures

**Module 03 · Functional Safety (ISO 26262)** · Notebook 4 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/03_Functional_Safety_ISO26262/04_fault_tree_analysis.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~75 min | [02 HARA](02_hara.ipynb), [03 FMEA and FMEDA](03_fmea_and_fmeda.ipynb); basic probability | ISO 26262-9:2018, ISO 26262-5:2018, IEC 61025:2006 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Distinguish **deductive** (top-down, FTA) from **inductive** (bottom-up, FMEA) safety analyses and place both within ISO 26262-9:2018 clause 8.
2. Read and draw the standard **fault tree symbols** and manipulate fault trees with **Boolean algebra**.
3. Build a fault tree for the AEB top event *"unintended full braking"* and compute its **minimal cut sets** with the MOCUS algorithm.
4. Quantify the top event **exactly**, with the **rare-event approximation**, and with a **Monte Carlo** check, starting from failure rates in FIT.
5. Rank basic events with **Birnbaum** and **Fussell–Vesely** importance measures.
6. Explain why **ASIL decomposition** (e.g. D → B(D) + B(D)) only works with **independence**, and quantify how **common-cause failures** (beta-factor model) erode redundancy.
7. Recognise where FTA reaches its limits for **machine-learning components**, whose performance insufficiencies have no meaningful failure rate.

## 1. Setup

The notebook uses only `numpy`, `pandas` and `matplotlib`; everything, including the fault tree engine, is written from scratch so that each step is visible.

In [ ]:
import math
import textwrap
from dataclasses import dataclass, field
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Circle, Rectangle, Polygon, FancyBboxPatch

rng = np.random.default_rng(42)
pd.set_option("display.width", 140)

# Okabe–Ito colour-blind-safe palette
OI = {"orange": "#E69F00", "sky": "#56B4E9", "green": "#009E73", "yellow": "#F0E442",
      "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7", "black": "#000000",
      "grey": "#999999"}
FIT = 1e-9  # 1 FIT = one failure in 10^9 device-hours
print("Setup complete.")

## 2. Deductive and inductive safety analyses

ISO 26262-9:2018 **clause 8 (Safety analyses)** describes two complementary directions of reasoning:

| | **Inductive (bottom-up)** | **Deductive (top-down)** |
|---|---|---|
| Question asked | *"If this part fails in this way, what happens?"* | *"What combinations of causes could produce this undesired event?"* |
| Typical method | FMEA / FMEDA ([notebook 03](03_fmea_and_fmeda.ipynb)) | Fault Tree Analysis (FTA, IEC 61025:2006) |
| Starting point | Every component and failure mode | One top event, usually the violation of a safety goal |
| Strength | Completeness over failure modes | Finds **combinations** of faults, shared causes and the minimal sets that are sufficient |
| Weakness | Poor at multiple-point failures | Only as complete as the analyst's imagination of causes |

ISO 26262 asks for both. In the safety-analysis tables of ISO 26262-4:2018 (system) and ISO 26262-5:2018 (hardware), inductive analysis is recommended at every ASIL, while deductive analysis is called for with increasing strength as the ASIL rises (most strongly for ASIL C and D). In practice the two meet: the FMEDA supplies failure modes and rates that become the **basic events** of the fault tree, and the fault tree shows which **combinations** of them violate the safety goal.

FTA can be **qualitative** (minimal cut sets, single points of failure, shared causes) or **quantitative** (probability of the top event). The qualitative result is often the more valuable one: a minimal cut set of size one is a single point of failure, whatever the numbers say.

## 3. Fault tree symbols and Boolean algebra

A fault tree is a Boolean expression drawn as a graph. Events are written in rectangles; the gate beneath a rectangle states how its inputs combine.

| Symbol | Name | Meaning |
|---|---|---|
| AND gate | $\;Y = X_1 \cdot X_2 \cdots X_n$ | output occurs only if **all** inputs occur |
| OR gate | $\;Y = X_1 + X_2 + \dots + X_n$ | output occurs if **at least one** input occurs |
| $k$/$n$ (voting) gate | $\;Y = \sum_{|S|=k} \prod_{i\in S} X_i$ | output occurs if **at least $k$ of $n$** inputs occur |
| Circle | basic event | a failure with its own probability or failure rate, not developed further |
| Diamond | undeveloped event | not developed further — lack of information, or outside the scope of this analysis |
| House | house event | an event that is normally expected to occur (a switch, e.g. "vehicle moving") |
| Triangle | transfer | the tree continues on another page |

Here "$\cdot$" is logical AND and "$+$" logical OR. The algebraic laws that matter most for fault trees are

$$
\text{idempotence: } A + A = A,\quad A\cdot A = A \qquad
\text{absorption: } A + A\cdot B = A \qquad
\text{distribution: } A\cdot(B + C) = A\cdot B + A\cdot C .
$$

Expanding a tree with distribution and simplifying with idempotence and absorption gives its **disjunctive normal form**: an OR of AND-terms. When no term contains another, the terms are the **minimal cut sets** (MCS): the smallest sets of basic events whose joint occurrence causes the top event.

In [ ]:
def gate_path(kind, x, y, w=0.9, h=0.7):
    # Return a matplotlib Path for an AND or OR gate centred at (x, y); output at the top.
    l, r, b, t = x - w / 2, x + w / 2, y - h / 2, y + h / 2
    if kind == "AND":
        verts = [(l, b), (r, b), (r, y), (r, t + 0.15 * h), (l, t + 0.15 * h), (l, y), (l, b)]
        codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    else:  # OR and k/n
        verts = [(l, b), (x, b + 0.35 * h), (r, b), (r, y), (x, t), (l, y), (l, b), (l, b)]
        codes = [Path.MOVETO, Path.CURVE3, Path.CURVE3, Path.CURVE3, Path.CURVE3, Path.CURVE3, Path.CURVE3, Path.CLOSEPOLY]
    return Path(verts, codes)

def draw_symbol(ax, kind, x, y, label=None, k=None, n=None, fc="white"):
    if kind in ("AND", "OR", "KOFN"):
        ax.add_patch(PathPatch(gate_path("AND" if kind == "AND" else "OR", x, y), fc=fc, ec="black", lw=1.5))
        if kind == "KOFN":
            ax.text(x, y - 0.05, f"{k}/{n}", ha="center", va="center", fontsize=9, fontweight="bold")
    elif kind == "basic":
        ax.add_patch(Circle((x, y), 0.32, fc=fc, ec="black", lw=1.5))
    elif kind == "undeveloped":
        ax.add_patch(Polygon([(x, y + 0.38), (x + 0.45, y), (x, y - 0.38), (x - 0.45, y)], fc=fc, ec="black", lw=1.5))
    elif kind == "house":
        ax.add_patch(Polygon([(x - 0.35, y - 0.35), (x + 0.35, y - 0.35), (x + 0.35, y + 0.1),
                              (x, y + 0.4), (x - 0.35, y + 0.1)], fc=fc, ec="black", lw=1.5))
    elif kind == "transfer":
        ax.add_patch(Polygon([(x - 0.38, y - 0.33), (x + 0.38, y - 0.33), (x, y + 0.38)], fc=fc, ec="black", lw=1.5))
    if label:
        ax.text(x, y - 0.75, label, ha="center", va="top", fontsize=9)

fig, ax = plt.subplots(figsize=(11, 2.6))
items = [("AND", "AND gate"), ("OR", "OR gate"), ("KOFN", "k-out-of-n\nvoting gate"), ("basic", "basic event"),
         ("undeveloped", "undeveloped\nevent"), ("house", "house event"), ("transfer", "transfer")]
colours = [OI["sky"], OI["orange"], OI["yellow"], OI["green"], OI["grey"], OI["purple"], "white"]
for i, ((kind, lab), c) in enumerate(zip(items, colours)):
    draw_symbol(ax, kind, 1.5 * i + 0.75, 0.6, label=lab, k=2, n=3, fc=c)
ax.set_xlim(0, 1.5 * len(items)); ax.set_ylim(-0.9, 1.2); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Fault tree symbols (after IEC 61025:2006)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print("Text summary: the figure shows, left to right, the AND, OR and k-out-of-n gates, "
      "and the basic, undeveloped, house and transfer event symbols.")

## 4. The item and the top event

We continue with the **AEB item** of [notebook 02 (HARA)](02_hara.ipynb): the automatic emergency braking function *including the brake request path* to the braking system. In notebook 02 the malfunctioning behaviour *unintended AEB deceleration* (MB-02) at motorway speed with closely following traffic (S3/E4/C3) leads to the safety goal **SG-01 "Avoid unintended AEB deceleration", ASIL D**. Here we analyse its E/E causes in detail.

**Top event (violation of SG-01):** *Unintended full braking* — the vehicle builds full braking pressure although no valid collision target exists.

**Simplified architecture (illustrative).**

```
 radar ─┐                         ┌─────────────── independent plausibility monitor (MCU B)
 camera ┼─► ADAS SoC (main, MCU A) ─► brake request ─► [AND: monitor must agree] ─► CAN ─► brake ECU ─► hydraulic unit
 dynamics┘                         └── 2-out-of-3 confirmation of the target (radar track, camera track, ego dynamics)
```

The main channel computes the brake request; an independent monitor lets it pass only if at least two of three confirmation sources agree that a target is present. (Notebook 02 places this plausibility channel in the brake ECU; where it is located does not change the logic of the tree, only which coupling factors the dependent failure analysis must examine.) The message to the brake ECU is protected by an end-to-end (E2E) check (CRC and counter).

> **Illustrative data:** all failure rates, exposure times and probabilities below are synthetic, chosen for teaching; they are not measurements of any product.

## 5. A small fault tree engine

### 5.1 From failure rate to probability

For a component with constant failure rate $\lambda$ (exponential lifetime) that is exposed for a time $\tau$ without being detected or repaired, the probability that the fault is present is

$$
q = 1 - e^{-\lambda \tau} \;\approx\; \lambda\tau \quad (\lambda\tau \ll 1).
$$

Failure rates in automotive hardware are quoted in **FIT** (failures in time): $1\,\text{FIT} = 10^{-9}\,\text{h}^{-1}$. The exposure time is the crucial engineering lever: a fault detected at every power-up is exposed for one drive cycle; an undetected (latent) fault is exposed for the vehicle lifetime.

### 5.2 Data structures and MOCUS

**MOCUS** (Method for Obtaining Cut Sets, described in the NRC *Fault Tree Handbook*, NUREG-0492) expands the tree top-down: start with the row $\{\text{TOP}\}$; repeatedly replace a gate in a row — an **AND** gate by all its inputs in the same row, an **OR** gate by one new row per input, a $k$/$n$ gate by one row per $k$-combination. When no gates remain, remove duplicates and apply **absorption** (drop every set that contains another).

In [ ]:
@dataclass
class Basic:
    # A basic or undeveloped event.
    name: str
    desc: str
    lam_fit: float | None = None   # failure rate in FIT
    tau_h: float | None = None     # exposure time in hours
    p: float | None = None         # fixed probability (overrides lambda/tau), e.g. probability per demand
    kind: str = "basic"            # 'basic' or 'undeveloped'

    def prob(self) -> float:
        if self.p is not None:
            return float(self.p)
        if self.lam_fit is None:          # undeveloped and not quantified
            return 0.0
        return 1.0 - math.exp(-self.lam_fit * FIT * self.tau_h)

@dataclass
class Gate:
    name: str
    desc: str
    gate: str                      # 'AND', 'OR' or 'KOFN'
    children: list
    k: int = 1

class FaultTree:
    def __init__(self, nodes, top):
        self.nodes = {n.name: n for n in nodes}
        self.top = top
        for n in nodes:                       # consistency check
            if isinstance(n, Gate):
                missing = [c for c in n.children if c not in self.nodes]
                if missing:
                    raise ValueError(f"gate {n.name} references unknown nodes {missing}")

    # ---------- structure ----------
    def basics(self):
        seen, out = set(), []
        def walk(name):
            nd = self.nodes[name]
            if isinstance(nd, Basic):
                if name not in seen:
                    seen.add(name); out.append(name)
            else:
                for c in nd.children:
                    walk(c)
        walk(self.top)
        return out

    def q(self, overrides=None):
        q = {b: self.nodes[b].prob() for b in self.basics()}
        q.update(overrides or {})
        return q

    # ---------- qualitative: MOCUS + absorption ----------
    def mocus(self):
        rows, done = [frozenset([self.top])], []
        while rows:
            row = rows.pop()
            g = next((x for x in row if isinstance(self.nodes[x], Gate)), None)
            if g is None:
                done.append(row); continue
            gate, rest = self.nodes[g], row - {g}
            if gate.gate == "AND":
                rows.append(rest | frozenset(gate.children))
            elif gate.gate == "OR":
                rows.extend(rest | {c} for c in gate.children)
            elif gate.gate == "KOFN":
                rows.extend(rest | frozenset(c) for c in combinations(gate.children, gate.k))
            else:
                raise ValueError(gate.gate)
        return minimise(done)

    # ---------- structure function on many states at once ----------
    def evaluate(self, X, idx, name=None):
        name = name or self.top
        nd = self.nodes[name]
        if isinstance(nd, Basic):
            return X[:, idx[name]]
        vals = np.stack([self.evaluate(X, idx, c) for c in nd.children])
        if nd.gate == "AND":
            return vals.all(axis=0)
        if nd.gate == "OR":
            return vals.any(axis=0)
        return vals.sum(axis=0) >= nd.k

    # ---------- quantitative ----------
    def exact(self, q=None):
        # Exact top-event probability by enumerating all 2^n states (n <= ~18).
        q = q or self.q()
        names = self.basics(); n = len(names)
        X = ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1).astype(bool)
        qv = np.array([q[b] for b in names])
        w = np.where(X, qv, 1 - qv).prod(axis=1)
        top = self.evaluate(X, {b: i for i, b in enumerate(names)})
        return float(w[top].sum())

    def monte_carlo(self, n_samples, rng, q=None):
        q = q or self.q()
        names = self.basics()
        qv = np.array([q[b] for b in names])
        X = rng.random((n_samples, len(names))) < qv
        hits = self.evaluate(X, {b: i for i, b in enumerate(names)})
        p = hits.mean()
        half = 1.96 * math.sqrt(max(p * (1 - p), 1e-300) / n_samples)
        return p, (max(p - half, 0.0), p + half), int(hits.sum())

def minimise(sets):
    # Remove duplicates and apply absorption (A + AB = A).
    out = []
    for s in sorted(set(sets), key=len):
        if not any(m <= s for m in out):
            out.append(s)
    return out

def cut_prob(cs, q):
    return float(np.prod([q[e] for e in cs]))

def rare_event(mcs, q):
    return sum(cut_prob(c, q) for c in mcs)

def min_cut_upper_bound(mcs, q):
    return 1 - float(np.prod([1 - cut_prob(c, q) for c in mcs]))

def inclusion_exclusion(mcs, q):
    # Exact P(union of MCS); returns final value and the partial sums (Bonferroni bounds).
    partial, total = [], 0.0
    for r in range(1, len(mcs) + 1):
        s = sum(cut_prob(frozenset().union(*combo), q) for combo in combinations(mcs, r))
        total += (-1) ** (r + 1) * s
        partial.append(total)
    return total, partial

print("Engine ready: Basic, Gate, FaultTree, MOCUS, exact enumeration, inclusion-exclusion, Monte Carlo.")

### 5.3 A sanity check on a textbook tree

Before trusting the engine, test it on $T = A\cdot(B + C) + B\cdot C$ whose minimal cut sets we can derive by hand: $\{A,B\}, \{A,C\}, \{B,C\}$ — a 2-out-of-3 structure in disguise.

In [ ]:
toy = FaultTree([
    Gate("T", "top", "OR", ["G1", "G2"]),
    Gate("G1", "A and (B or C)", "AND", ["A", "G3"]),
    Gate("G3", "B or C", "OR", ["B", "C"]),
    Gate("G2", "B and C", "AND", ["B", "C"]),
    Basic("A", "a", p=0.1), Basic("B", "b", p=0.1), Basic("C", "c", p=0.1),
], top="T")
toy_mcs = toy.mocus()
q_toy = toy.q()
exact_2oo3 = 3 * 0.1 ** 2 - 2 * 0.1 ** 3          # closed form for 2-out-of-3 with q = 0.1
print("MCS:", sorted(sorted(c) for c in toy_mcs))
print(f"exact (enumeration)       = {toy.exact():.6f}")
print(f"exact (incl.-excl.)       = {inclusion_exclusion(toy_mcs, q_toy)[0]:.6f}")
print(f"closed form 3q^2 - 2q^3   = {exact_2oo3:.6f}")
print(f"rare-event approximation  = {rare_event(toy_mcs, q_toy):.6f}")
assert abs(toy.exact() - exact_2oo3) < 1e-12

## 6. The AEB fault tree

Each basic event carries a failure rate and an exposure time. Monitor faults are **latent**: without a self-test they persist undetected, so their exposure time is the vehicle operating lifetime, here taken as $T_\text{life} = 10\,000\,\text{h}$. The perception false positive is drawn as an **undeveloped** event: it is a performance insufficiency of the machine-learning component, not a random hardware fault, and has no failure rate (Section 10).

In [ ]:
T_LIFE = 10_000.0   # vehicle operating lifetime in hours (assumption)
T_DRIVE = 1.0       # one drive cycle in hours (assumption)

def aeb_nodes(t_mon=T_LIFE):
    return [
        Gate("TOP", "Unintended full braking", "OR", ["G_REQ", "G_BUS", "E_BRK"]),
        Gate("G_REQ", "Erroneous request passes the monitor", "AND", ["G_MAIN", "G_MON"]),
        Gate("G_MAIN", "Main channel requests full braking", "OR", ["E_MCU_A", "E_RAD", "U_PERC"]),
        Gate("G_MON", "Monitor fails to block the request", "OR", ["E_MCU_B", "G_VOTE"]),
        Gate("G_VOTE", "2 of 3 sources falsely confirm a target", "KOFN", ["E_C_RAD", "E_C_CAM", "E_C_DYN"], k=2),
        Gate("G_BUS", "Corrupted command accepted by brake ECU", "AND", ["E_BUS", "E_E2E"]),
        Basic("E_MCU_A", "ADAS SoC random HW fault", lam_fit=100, tau_h=T_LIFE),
        Basic("E_RAD", "Radar HW fault creates ghost target", lam_fit=50, tau_h=T_LIFE),
        Basic("U_PERC", "Perception false positive (ML)", kind="undeveloped"),
        Basic("E_MCU_B", "Monitor MCU latent fault", lam_fit=100, tau_h=t_mon),
        Basic("E_C_RAD", "Radar track confirmation wrong", lam_fit=200, tau_h=T_LIFE),
        Basic("E_C_CAM", "Camera track confirmation wrong", lam_fit=200, tau_h=T_LIFE),
        Basic("E_C_DYN", "Ego-dynamics plausibility wrong", lam_fit=200, tau_h=T_LIFE),
        Basic("E_BUS", "CAN brake message corrupted", lam_fit=50, tau_h=T_LIFE),
        Basic("E_E2E", "E2E check misses corruption", p=1e-6),
        Basic("E_BRK", "Brake ECU builds pressure spuriously", lam_fit=0.1, tau_h=T_LIFE),
    ]

aeb = FaultTree(aeb_nodes(), top="TOP")
q_aeb = aeb.q()
basic_tbl = pd.DataFrame([{
    "event": b, "description": aeb.nodes[b].desc, "kind": aeb.nodes[b].kind,
    "lambda [FIT]": aeb.nodes[b].lam_fit, "tau [h]": aeb.nodes[b].tau_h,
    "q": q_aeb[b]} for b in aeb.basics()])
display(basic_tbl.style.format({"q": "{:.2e}", "lambda [FIT]": "{:g}", "tau [h]": "{:g}"}, na_rep="—"))

### 6.1 Drawing the tree

The layout is a simple tidy-tree algorithm: leaves are spaced evenly from left to right, each gate sits above the mean position of its inputs.

In [ ]:
def layout(ft, name=None, depth=0, pos=None, counter=None, path=()):
    # Return a list of (occurrence_id, node_name, x, y, parent_occurrence_id).
    if pos is None:
        pos, counter = [], [0]
    name = name or ft.top
    nd = ft.nodes[name]
    occ = path + (name,)
    if isinstance(nd, Basic):
        x = counter[0]; counter[0] += 1
    else:
        xs = [layout(ft, c, depth + 1, pos, counter, occ) for c in nd.children]
        x = float(np.mean(xs))
    pos.append((occ, name, x, -depth, path if path else None))
    return x if depth else pos

def draw_tree(ft, q=None, title="", highlight=(), figsize=(19, 9.5)):
    q = q or ft.q()
    pos = layout(ft)
    sx, sy = 2.15, 2.6
    coords = {occ: (x * sx, y * sy) for occ, _, x, y, _ in pos}
    fig, ax = plt.subplots(figsize=figsize)
    for occ, name, x, y, parent in pos:          # connectors first
        if parent is not None:
            (x0, y0), (x1, y1) = coords[parent], coords[occ]
            gy = y0 - 1.35                     # bottom of parent's gate
            ax.plot([x0, x0, x1, x1], [gy, gy - 0.25, gy - 0.25, y1 + 0.45], color="black", lw=1)
    for occ, name, x, y, parent in pos:
        X, Y = coords[occ]
        nd = ft.nodes[name]
        is_gate = isinstance(nd, Gate)
        fc = "#FFFFFF"
        if name in highlight:
            fc = "#FCE3D0"
        ax.add_patch(Rectangle((X - 0.98, Y - 0.45), 1.96, 0.9, fc=fc, ec="black", lw=1.2))
        label = "\n".join(textwrap.wrap(nd.desc, 20)[:3])
        ax.text(X, Y + 0.02, label, ha="center", va="center", fontsize=7.4)
        ax.text(X - 0.95, Y + 0.56, name, ha="left", va="bottom", fontsize=7, color=OI["blue"], fontweight="bold")
        sym_y = Y - 0.85
        ax.plot([X, X], [Y - 0.45, sym_y + 0.3], color="black", lw=1)
        if is_gate:
            colour = {"AND": OI["sky"], "OR": OI["orange"], "KOFN": OI["yellow"]}[nd.gate]
            draw_symbol(ax, nd.gate, X, sym_y - 0.1, k=nd.k, n=len(nd.children), fc=colour)
        else:
            kind = nd.kind
            draw_symbol(ax, kind, X, sym_y - 0.1, fc=OI["green"] if kind == "basic" else OI["grey"])
            qtxt = f"q={q[name]:.1e}" if nd.lam_fit is not None or nd.p is not None else "not quantified"
            ax.text(X, sym_y - 0.62, qtxt, ha="center", va="top", fontsize=7)
    xs = [c[0] for c in coords.values()]; ys = [c[1] for c in coords.values()]
    ax.set_xlim(min(xs) - 1.2, max(xs) + 1.2); ax.set_ylim(min(ys) - 2.0, max(ys) + 1.0)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

draw_tree(aeb, q_aeb, title="Fault tree: unintended full braking (AEB, illustrative)")
n_g = sum(isinstance(aeb.nodes[n], Gate) for n in aeb.nodes)
print(f"Text summary: the tree has {n_g} gates and {len(aeb.basics())} basic/undeveloped events. "
      "TOP is an OR of (i) an erroneous request that also passes the monitor (AND), "
      "(ii) a corrupted bus command accepted by the brake ECU (AND) and (iii) a spurious brake-ECU fault.")

### 6.2 Minimal cut sets

In [ ]:
mcs = aeb.mocus()
mcs_tbl = pd.DataFrame([{"order": len(c), "cut set": " · ".join(sorted(c)),
                         "P(cut set)": cut_prob(c, q_aeb)} for c in mcs])
mcs_tbl = mcs_tbl.sort_values(["order", "P(cut set)"], ascending=[True, False]).reset_index(drop=True)
display(mcs_tbl.style.format({"P(cut set)": "{:.2e}"}))

orders = mcs_tbl["order"].value_counts().sort_index()
spof = [c for c in mcs if len(c) == 1]
print(f"{len(mcs)} minimal cut sets; by order: " + ", ".join(f"order {k}: {v}" for k, v in orders.items()))
print("Single points of failure:", ", ".join(sorted(next(iter(c)) for c in spof)) or "none")

The single-order cut set is the brake ECU's own residual fault: no amount of redundancy in the ADAS domain can protect against it, which is why the brake ECU carries its own internal safety mechanisms (and why its residual rate is so small). Every path through the ADAS domain needs **two** independent faults — that is the purpose of the monitor.

## 7. Quantification

Three ways to compute $Q_\text{TOP} = P\big(\bigcup_j C_j\big)$ over the minimal cut sets $C_j$:

* **Exact** — by enumerating all $2^n$ states of the basic events, or by **inclusion–exclusion**
  $$Q = \sum_j P(C_j) - \sum_{j<l} P(C_j \cap C_l) + \sum_{j<l<m} P(C_j\cap C_l\cap C_m) - \dots,\qquad P(C_j\cap C_l) = \prod_{i \in C_j\cup C_l} q_i .$$
  Note the **union** of events in the intersection term: a basic event shared by two cut sets is counted once.
* **Rare-event approximation** — keep only the first sum, $Q \approx \sum_j \prod_{i\in C_j} q_i$. It is an upper bound and excellent when all $q_i \ll 1$.
* **Monte Carlo** — sample the basic-event states and evaluate the tree; useful for large trees and as an independent check.

In [ ]:
Q_exact = aeb.exact(q_aeb)
Q_ie, partial = inclusion_exclusion(mcs, q_aeb)
Q_rare = rare_event(mcs, q_aeb)
Q_mcub = min_cut_upper_bound(mcs, q_aeb)

res = pd.DataFrame({
    "method": ["exact (state enumeration)", "exact (inclusion-exclusion)", "min-cut upper bound", "rare-event approximation"],
    "Q_TOP over lifetime": [Q_exact, Q_ie, Q_mcub, Q_rare]})
res["relative error vs exact"] = res["Q_TOP over lifetime"] / Q_exact - 1
display(res.style.format({"Q_TOP over lifetime": "{:.6e}", "relative error vs exact": "{:+.2e}"}))
print(f"Inclusion-exclusion over {len(mcs)} cut sets needed {2 ** len(mcs) - 1:,} intersection terms; "
      f"it agrees with state enumeration to {abs(Q_exact - Q_ie) / Q_exact:.1e} (relative). "
      f"The rare-event approximation overestimates by {Q_rare / Q_exact - 1:.1e} (relative).")

### 7.1 Monte Carlo — and why it struggles with rare events

At $Q \approx 10^{-5}$ a plain Monte Carlo run of a million samples sees only a handful of top events. We therefore check the engine at **accelerated** failure rates (all $\lambda$ multiplied by a factor $s$), where the top event is common enough to sample — and we watch the rare-event approximation lose accuracy as the $q_i$ grow.

In [ ]:
def scaled_q(ft, s):
    q = {}
    for b in ft.basics():
        nd = ft.nodes[b]
        if nd.p is not None:
            q[b] = nd.p
        elif nd.lam_fit is None:
            q[b] = 0.0
        else:
            q[b] = 1 - math.exp(-s * nd.lam_fit * FIT * nd.tau_h)
    return q

N_MC = 1_000_000
rows = []
for s in [1, 30, 100, 300, 1000]:
    qs = scaled_q(aeb, s)
    ex = aeb.exact(qs)
    mc, (lo, hi), hits = aeb.monte_carlo(N_MC, rng, qs)
    rows.append({"scale s": s, "exact": ex, "rare-event": rare_event(mcs, qs),
                 "MC estimate": mc, "MC 95% CI low": lo, "MC 95% CI high": hi, "MC hits": hits})
mc_tbl = pd.DataFrame(rows)
mc_tbl["exact inside CI"] = (mc_tbl["MC 95% CI low"] <= mc_tbl["exact"]) & (mc_tbl["exact"] <= mc_tbl["MC 95% CI high"])
mc_tbl["rare-event rel. error"] = mc_tbl["rare-event"] / mc_tbl["exact"] - 1
display(mc_tbl.style.format({c: "{:.3e}" for c in ["exact", "rare-event", "MC estimate", "MC 95% CI low", "MC 95% CI high"]}
                            | {"rare-event rel. error": "{:+.2%}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.errorbar(mc_tbl["scale s"], mc_tbl["MC estimate"],
            yerr=[mc_tbl["MC estimate"] - mc_tbl["MC 95% CI low"], mc_tbl["MC 95% CI high"] - mc_tbl["MC estimate"]],
            fmt="o", color=OI["vermillion"], capsize=4, label=f"Monte Carlo (N={N_MC:,}), 95% CI")
ax.plot(mc_tbl["scale s"], mc_tbl["exact"], "-", color=OI["blue"], label="exact")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("failure-rate scale factor s [-]"); ax.set_ylabel("Q_TOP over lifetime [-]")
ax.set_title("Monte Carlo check of the exact result"); ax.legend(fontsize=8)
ax = axes[1]
ax.plot(mc_tbl["scale s"], 100 * mc_tbl["rare-event rel. error"], "s-", color=OI["green"])
ax.set_xscale("log"); ax.set_xlabel("failure-rate scale factor s [-]"); ax.set_ylabel("overestimate [%]")
ax.set_title("Rare-event approximation error"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Exact value inside the Monte Carlo 95% CI in {int(mc_tbl['exact inside CI'].sum())} of {len(mc_tbl)} runs. "
      f"At s=1 Monte Carlo saw only {int(mc_tbl.loc[0, 'MC hits'])} top events in {N_MC:,} samples. "
      f"The rare-event overestimate grows from {mc_tbl['rare-event rel. error'].iloc[0]:.2e} at s=1 "
      f"to {mc_tbl['rare-event rel. error'].iloc[-1]:.1%} at s={int(mc_tbl['scale s'].iloc[-1])}.")

### 7.2 Importance measures

Which basic events matter most? Two classic measures:

* **Birnbaum importance** — the sensitivity of the top event to event $i$:
  $$I_B(i) = \frac{\partial Q}{\partial q_i} = Q(q_i{=}1) - Q(q_i{=}0).$$
  It tells you how much the system depends on component $i$ *irrespective of how reliable $i$ is*.
* **Fussell–Vesely importance** — the fraction of top-event probability that involves event $i$:
  $$I_{FV}(i) = \frac{P\big(\bigcup_{j:\, i\in C_j} C_j\big)}{Q} \;\approx\; \frac{\sum_{j:\, i \in C_j} P(C_j)}{Q}.$$
  It tells you where improvement pays off.

In [ ]:
def importance(ft, mcs, q):
    Q = ft.exact(q)
    names = ft.basics(); n = len(names); idx = {b: i for i, b in enumerate(names)}
    X = ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1).astype(bool)
    qv = np.array([q[b] for b in names])
    w = np.where(X, qv, 1 - qv).prod(axis=1)
    cut_true = {c: X[:, [idx[e] for e in c]].all(axis=1) for c in mcs}
    rows = []
    for b in names:
        if q[b] == 0.0:
            continue
        birn = ft.exact({**q, b: 1.0}) - ft.exact({**q, b: 0.0})
        involved = np.zeros(len(X), dtype=bool)
        for c, m in cut_true.items():
            if b in c:
                involved |= m
        rows.append({"event": b, "q": q[b], "Birnbaum": birn, "Fussell-Vesely": w[involved].sum() / Q})
    return pd.DataFrame(rows).sort_values("Fussell-Vesely", ascending=False).reset_index(drop=True)

imp = importance(aeb, mcs, q_aeb)
display(imp.style.format({"q": "{:.2e}", "Birnbaum": "{:.2e}", "Fussell-Vesely": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
axes[0].barh(imp["event"], imp["Fussell-Vesely"], color=OI["blue"])
axes[0].set_xscale("log"); axes[0].set_xlabel("Fussell–Vesely importance [-]"); axes[0].invert_yaxis()
axes[0].set_title("Share of top-event probability")
axes[1].barh(imp["event"], imp["Birnbaum"], color=OI["orange"])
axes[1].set_xscale("log"); axes[1].set_xlabel("Birnbaum importance dQ/dq_i [-]")
axes[1].set_title("Sensitivity of top event")
plt.tight_layout(); plt.show()

top_fv = imp.iloc[0]; top_b = imp.sort_values("Birnbaum", ascending=False).iloc[0]
print(f"Highest Fussell–Vesely: {top_fv['event']} ({top_fv['Fussell-Vesely']:.1%} of Q_TOP). "
      f"Highest Birnbaum: {top_b['event']} (dQ/dq = {top_b['Birnbaum']:.2e}).")

The two rankings answer different questions. The brake ECU fault has the largest Birnbaum importance ($\partial Q/\partial q = 1$) because it is a single point of failure: the top event follows it one-to-one, so its very low failure rate must be backed by strong evidence. Fussell–Vesely, by contrast, highlights the events that actually *contribute* most of $Q_\text{TOP}$ — here the latent monitor fault and the main-channel faults — and therefore where design improvement pays off. (Fussell–Vesely values need not sum to one, because cut sets overlap.)

### 7.3 Latent faults and the value of a self-test

The monitor's own faults are latent: they do nothing until the main channel misbehaves. Reducing their exposure time — e.g. with a power-up self-test so that $\tau_\text{mon} = T_\text{drive}$ rather than $T_\text{life}$ — cuts the dual-point contribution. This is the intuition behind the **latent-fault metric (LFM)** of ISO 26262-5.

In [ ]:
taus = np.logspace(0, 4, 9)
Qs = [FaultTree(aeb_nodes(t_mon=t), "TOP").exact() for t in taus]
q_spf = q_aeb["E_BRK"]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.loglog(taus, Qs, "o-", color=OI["purple"], label="Q_TOP (monitor MCU exposure varied)")
ax.axhline(q_spf, ls="--", color=OI["grey"], label="single-point contribution (E_BRK)")
ax.set_xlabel("exposure time of monitor MCU fault tau_mon [h]"); ax.set_ylabel("Q_TOP over lifetime [-]")
ax.set_title("Effect of latent-fault exposure time on the top event"); ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()
print(f"Q_TOP falls from {Qs[-1]:.3e} (latent for {taus[-1]:.0f} h) to {Qs[0]:.3e} (tested every {taus[0]:.0f} h drive cycle); "
      f"the floor is set by the single-point fault ({q_spf:.2e}).")

## 8. ASIL decomposition: why independence is the whole point

ISO 26262-9:2018 **clause 5** allows a safety requirement of a given ASIL to be decomposed into redundant requirements allocated to **sufficiently independent** elements. For ASIL D the permitted decompositions are C(D)+A(D), **B(D)+B(D)** and D(D)+QM(D); the ASIL of the original goal is kept in brackets because the hardware metrics and the PMHF (Section 9) are still evaluated against ASIL D.

Notebook 02 decomposes the ASIL D requirement derived from SG-01 in exactly this way (decision channel B(D) + plausibility channel B(D)). In our tree the main channel and the monitor form such a pair: the erroneous request reaches the brakes only if **both** fail (the AND gate `G_REQ`). The decomposition is valid only if the two channels have no common cause — which ISO 26262-9:2018 **clause 7** requires to be shown by a **dependent failure analysis (DFA)** covering common-cause and cascading failures.

> The same table is available as `avsafety.iso26262.decomposition_options` (see repo `avsafety/`).

In [ ]:
DECOMPOSITIONS = {"D": [("C", "A"), ("B", "B"), ("D", "QM")], "C": [("B", "A"), ("C", "QM")],
                  "B": [("A", "A"), ("B", "QM")], "A": [("A", "QM")]}   # ISO 26262-9:2018 clause 5

def decomposition_options(asil):
    return [(f"{a}({asil})", f"{b}({asil})") for a, b in DECOMPOSITIONS[asil]]

print("Permitted decompositions of ASIL D:", decomposition_options("D"))
print("Chosen for the AEB example: main channel B(D) + plausibility monitor B(D)")

### 8.1 The beta-factor model

The simplest quantitative model of common-cause failure (CCF) splits each channel's failure rate into an independent part and a common part shared with its redundant partner (the approach used, for example, in IEC 61508-6:2010 Annex D):

$$
\lambda = \lambda_\text{ind} + \lambda_\text{CCF}, \qquad \lambda_\text{CCF} = \beta\,\lambda .
$$

For two identical channels with $q_\text{ind} = 1 - e^{-(1-\beta)\lambda T}$ and $q_\text{CCF} = 1 - e^{-\beta\lambda T}$:

$$
Q_\text{pair} = q_\text{CCF} + (1 - q_\text{CCF})\, q_\text{ind}^2 \;\approx\; \beta\lambda T + \big((1-\beta)\lambda T\big)^2 .
$$

The redundant pair is only better than one channel by the factor $\approx \lambda T$ when $\beta = 0$; as soon as $\beta \gtrsim \lambda T$, the common-cause term dominates.

In [ ]:
lam, T = 100 * FIT, T_LIFE
betas = np.concatenate([[0], np.logspace(-5, -0.7, 60)])
q_single = 1 - math.exp(-lam * T)
def q_pair(beta):
    q_ind = 1 - np.exp(-(1 - beta) * lam * T); q_ccf = 1 - np.exp(-beta * lam * T)
    return q_ccf + (1 - q_ccf) * q_ind ** 2, q_ccf, (1 - q_ccf) * q_ind ** 2
Qp, Qccf, Qind = q_pair(betas)

fig, ax = plt.subplots(figsize=(8, 4.4))
b = betas[1:]
ax.loglog(b, Qp[1:], color=OI["blue"], lw=2.2, label="redundant pair, total")
ax.loglog(b, Qccf[1:], "--", color=OI["vermillion"], label="common-cause term")
ax.loglog(b, Qind[1:], ":", color=OI["green"], lw=2, label="independent double-fault term")
ax.axhline(q_single, color=OI["grey"], ls="-.", label="single channel")
for beta_ref in (0.01, 0.05, 0.1):
    ax.axvline(beta_ref, color="black", lw=0.6, alpha=0.4)
    ax.text(beta_ref, 2e-8, f"beta={beta_ref:g}", rotation=90, fontsize=7, va="bottom", ha="right")
ax.set_xlabel("beta factor (fraction of failures that are common cause) [-]"); ax.set_ylabel("probability over lifetime [-]")
ax.set_title(f"Redundancy versus common cause (lambda = {lam / FIT:.0f} FIT, T = {T:.0f} h)")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

beta_star = b[np.argmax(Qccf[1:] > Qind[1:])]
for beta_ref in (0.0, 0.01, 0.05):
    tot = q_pair(np.array([beta_ref]))[0][0]
    print(f"beta={beta_ref:<5g}: Q_pair = {tot:.2e}  -> benefit over single channel x{q_single / tot:,.0f}")
print(f"The common-cause term exceeds the independent term once beta > {beta_star:.1e} (about lambda*T = {lam * T:.1e}).")

### 8.2 Making a dependency visible in the tree

Beta-factors are a statistical shortcut. In a fault tree it is better to **model the shared cause explicitly**. Suppose both channels are fed by the same voltage regulator, and an over-voltage fault makes both processors behave erratically. The event `E_PSU` then appears under *both* `G_MAIN` and `G_MON`. MOCUS needs no special treatment: the shared event simply ends up in both branches of the AND gate, and absorption turns it into a **single-order cut set**.

In [ ]:
def aeb_nodes_shared_psu(lam_psu=10):
    nodes = aeb_nodes()
    for n in nodes:
        if n.name in ("G_MAIN", "G_MON"):
            n.children = n.children + ["E_PSU"]
    nodes.append(Basic("E_PSU", "Shared supply over-voltage", lam_fit=lam_psu, tau_h=T_LIFE))
    return nodes

aeb_psu = FaultTree(aeb_nodes_shared_psu(), "TOP")
mcs_psu = aeb_psu.mocus()
Q_psu = aeb_psu.exact()
new_spof = sorted(next(iter(c)) for c in mcs_psu if len(c) == 1)
print("Single points of failure with shared supply:", new_spof)
print(f"Q_TOP independent channels : {Q_exact:.3e}")
print(f"Q_TOP shared supply (10 FIT): {Q_psu:.3e}  (x{Q_psu / Q_exact:.1f})")
draw_tree(aeb_psu, title="Shared power supply E_PSU appears in both redundant branches", highlight=("E_PSU",),
          figsize=(21, 9.5))

**Shared software is worse.** If both channels run the same object-fusion library, a defect in it is a **systematic** fault: it will trigger in *every* vehicle whenever its triggering condition occurs. It has no failure rate, so the beta-factor model cannot capture it. ISO 26262-9 clause 7 addresses such coupling factors through the DFA and measures such as **diversity** (different algorithms, tools, teams or sensors), **separation** and **freedom from interference** — arguments, not numbers.

Typical coupling factors to look for in a DFA: shared power supply, clock and ground; shared sensor inputs; shared software components and libraries; same development tool or team; shared physical location (temperature, vibration, water ingress); shared communication bus.

## 9. From the fault tree to PMHF intuition

ISO 26262-5:2018 (clause 9) requires the **probabilistic metric for random hardware failures (PMHF)** — an average probability per hour of violating the safety goal due to random hardware faults — to be below a target, $10^{-8}\,\text{h}^{-1}$ for ASIL D. [Notebook 03](03_fmea_and_fmeda.ipynb) computes it from FMEDA rows; the fault tree gives a complementary view. A crude average rate over the lifetime is

$$
\overline{\lambda}_\text{TOP} \approx \frac{Q_\text{TOP}(T_\text{life})}{T_\text{life}} .
$$

This is intuition only: the PMHF evaluation in the standard distinguishes single-point, residual and latent multiple-point faults and their diagnostic coverage. Note also that ASIL decomposition does **not** relax the PMHF target — it remains that of the original ASIL D goal.

In [ ]:
PMHF_TARGET_PER_HOUR = {"B": 1e-7, "C": 1e-7, "D": 1e-8}   # ISO 26262-5:2018, same as avsafety.iso26262

summary = pd.DataFrame({
    "architecture": ["independent channels", "shared supply (10 FIT)"],
    "Q_TOP over lifetime": [Q_exact, Q_psu]})
summary["average rate [1/h]"] = summary["Q_TOP over lifetime"] / T_LIFE
summary["meets ASIL D target (1e-8/h)"] = summary["average rate [1/h]"] < PMHF_TARGET_PER_HOUR["D"]
spf_share = cut_prob(frozenset(["E_BRK"]), q_aeb) / Q_exact
display(summary.style.format({"Q_TOP over lifetime": "{:.3e}", "average rate [1/h]": "{:.2e}"}))
for _, r in summary.iterrows():
    verdict = "below" if r["meets ASIL D target (1e-8/h)"] else "above"
    print(f"{r['architecture']}: {r['average rate [1/h]']:.2e} /h, {verdict} the ASIL D target.")
print(f"The single-point fault of the brake ECU accounts for {spf_share:.1%} of Q_TOP in the independent case.")

## 10. Where fault trees meet machine learning

The event `U_PERC` — *the perception network reports a pedestrian that is not there* — was left undeveloped. This is not laziness:

* A random hardware fault has a physical failure rate that is (approximately) constant and transferable between vehicles. A **false positive of a neural network** is not a fault of the item in the ISO 26262 sense: the network does what it was trained to do, and fails in particular **scenarios** (glare, reflections, unusual objects). Its frequency depends on the operational design domain and the mix of situations, not on time.
* Such **performance insufficiencies** and their **triggering conditions** are the subject of ISO 21448:2022 (SOTIF, [Module 04](../04_SOTIF_ISO21448/README.md)); ML-specific causes — data, training, distribution shift — are treated in ISO/PAS 8800:2024 ([Module 06](../06_AI_Safety_ISO_PAS_8800/README.md)).
* The fault tree is still useful *qualitatively*: it shows where an ML output enters the architecture, and whether redundancy protects against it.

The what-if below makes the last point. We assign `U_PERC` a placeholder probability — *not a failure rate, but a stand-in for the residual risk that SOTIF validation would have to bound* — and compare an **independent** monitor with one whose camera confirmation reuses the **same** perception output.

In [ ]:
def aeb_nodes_ml(shared_perception):
    nodes = aeb_nodes()
    for n in nodes:
        if n.name == "U_PERC":
            n.p = P_PERC
        if shared_perception and n.name == "G_MON":
            n.children = n.children + ["U_PERC"]   # monitor's camera confirmation reuses the same network output
    return nodes

rows = []
for P_PERC in [1e-4, 1e-3, 1e-2]:
    for shared in (False, True):
        ft = FaultTree(aeb_nodes_ml(shared), "TOP")
        rows.append({"placeholder P(U_PERC)": P_PERC, "monitor input": "shared perception" if shared else "independent",
                     "Q_TOP": ft.exact(), "U_PERC is single point": any(c == frozenset(["U_PERC"]) for c in ft.mocus())})
ml_tbl = pd.DataFrame(rows)
display(ml_tbl.style.format({"placeholder P(U_PERC)": "{:.0e}", "Q_TOP": "{:.3e}"}))
worst = ml_tbl[ml_tbl["monitor input"] == "shared perception"]["Q_TOP"].max()
best = ml_tbl[ml_tbl["monitor input"] == "independent"]["Q_TOP"].max()
print(f"At the largest placeholder value, sharing the perception output raises Q_TOP from {best:.2e} to {worst:.2e} "
      f"(x{worst / best:,.0f}) because U_PERC becomes a single-order cut set.")

The lesson: an ML false positive is a **common-cause initiator** for every channel that consumes the same output. Independence for ML components therefore means **diverse inputs and models**, and the probability of the remaining insufficiency must be argued with SOTIF evidence (validation targets, triggering-condition analysis) and ISO/PAS 8800 evidence (data, uncertainty, runtime monitoring) — not with a FIT number. The [safety-case notebook](../08_Standards_Regulation_and_Safety_Case/04_safety_case_gsn.ipynb) shows how these strands of evidence are combined into one argument.

## 11. Exercises

**Exercise 1 — absorption by hand.** Simplify $T = A\cdot B + A\cdot B\cdot C + A\cdot(B + D)$ to its minimal cut sets, then check with the engine.

<details><summary>Answer</summary>

$A\cdot(B+D) = AB + AD$, so $T = AB + ABC + AB + AD$. Idempotence removes the duplicate $AB$, absorption removes $ABC$ (it contains $AB$): MCS $= \{A,B\}, \{A,D\}$.

```python
ex = FaultTree([Gate("T","","OR",["G1","G2","G3"]), Gate("G1","","AND",["A","B"]),
                Gate("G2","","AND",["A","B","C"]), Gate("G3","","AND",["A","G4"]),
                Gate("G4","","OR",["B","D"])] + [Basic(x, x, p=0.1) for x in "ABCD"], "T")
print(sorted(sorted(c) for c in ex.mocus()))   # [['A', 'B'], ['A', 'D']]
```
</details>

**Exercise 2 — voting logic.** Replace the 2-out-of-3 confirmation gate by a 1-out-of-3 gate (the monitor passes the request if *any* source confirms). How do the minimal cut sets and $Q_\text{TOP}$ change, and what does that say about the design?

<details><summary>Answer</summary>

```python
nodes = aeb_nodes()
for n in nodes:
    if n.name == "G_VOTE": n.k = 1
ft1 = FaultTree(nodes, "TOP")
print(len(ft1.mocus()), ft1.exact() / Q_exact)
```
All order-3 cut sets collapse to order 2 (each main-channel fault plus any single confirmation fault), and $Q_\text{TOP}$ rises. A 1oo3 confirmation is permissive: good for availability of braking, poor for preventing unintended braking. The voting threshold trades the two hazards (missed versus unintended braking) against each other.
</details>

**Exercise 3 — how large may beta be?** With $\lambda = 100$ FIT per channel and $T_\text{life}=10\,000$ h, find the largest $\beta$ for which the redundant pair still achieves an average rate below $10^{-9}\,\text{h}^{-1}$ (a typical budget allocated to this part of the architecture).

<details><summary>Answer</summary>

```python
from scipy.optimize import brentq
f = lambda beta: q_pair(np.array([beta]))[0][0] / T_LIFE - 1e-9
print(brentq(f, 1e-9, 0.5))
```
The independent term alone gives $\approx (10^{-3})^2/10^4 = 10^{-10}\,\text{h}^{-1}$, so the common-cause budget is $\approx 9\times10^{-10}\,\text{h}^{-1}$, i.e. $\beta\lambda \lesssim 0.9$ FIT and $\beta \lesssim 0.009$. Values of $\beta$ below 1 % require strong separation and diversity arguments.
</details>

## Key takeaways

1. **FTA is deductive**: it starts from a safety-goal violation and finds the combinations of causes; FMEA/FMEDA is inductive. ISO 26262 expects both; the weight given to deductive analysis grows with the ASIL.
2. **Minimal cut sets** are the qualitative heart of FTA; a cut set of order one is a single point of failure regardless of the numbers.
3. **Quantification** needs failure rates *and* exposure times; $q = 1 - e^{-\lambda\tau}$. The rare-event approximation is accurate for small $q$, exact methods (enumeration, inclusion–exclusion) confirm it, and plain Monte Carlo is inefficient for rare top events.
4. **Importance measures** (Birnbaum, Fussell–Vesely) show where design effort and evidence matter most.
5. **ASIL decomposition** (D → B(D)+B(D)) is only valid with demonstrated **independence**; a few per cent of common-cause failures (beta-factor) can erase the benefit of redundancy, and shared software defects are systematic and cannot be expressed as a rate.
6. **ML performance insufficiencies** have no meaningful failure rate; FTA shows *where* they enter the architecture, SOTIF and ISO/PAS 8800 must bound *how often*.

## Self-assessment

1. Why is FTA called a *deductive* analysis, and which question does it answer that FMEA cannot answer easily?
2. Derive the minimal cut sets of $T = (A + B)\cdot(A + C)$.
3. A basic event has $\lambda = 50$ FIT and is latent for $10\,000$ h. What is $q$? What if it is tested at every 1 h drive cycle?
4. When does the rare-event approximation overestimate the top-event probability noticeably?
5. Explain the difference between Birnbaum and Fussell–Vesely importance with an example.
6. Why does ASIL decomposition require a dependent failure analysis, and why does it not change the PMHF target?
7. Why can a neural-network false positive not be given a FIT rate?

<details><summary>Answers</summary>

1. It reasons from the effect (top event) to causes. It systematically finds **combinations** of faults (multiple-point failures) and shared causes, which a component-by-component FMEA tends to miss.
2. Distribution: $AA + AC + BA + BC = A + AC + AB + BC$; absorption leaves $A + BC$, so MCS $= \{A\}, \{B, C\}$.
3. $q = 1 - e^{-50\cdot10^{-9}\cdot10^4} \approx 5.0\times10^{-4}$; with $\tau = 1$ h, $q \approx 5\times10^{-8}$.
4. When basic-event probabilities are no longer small (roughly $q \gtrsim 0.01$–$0.1$) or many cut sets overlap, so the neglected intersection terms become significant.
5. Birnbaum: sensitivity $\partial Q/\partial q_i$, independent of $q_i$ itself (a very reliable brake ECU can have high Birnbaum importance because it is a single point). Fussell–Vesely: the fraction of $Q$ to which event $i$ contributes — high for events that are both critical and comparatively likely.
6. Decomposition relies on the redundant elements failing independently; a common cause would let a single fault violate the original ASIL D goal. The random-hardware target belongs to the safety goal, which still has ASIL D.
7. It is not a random physical fault but a functional insufficiency that occurs in specific scenarios; its frequency depends on the ODD and the scenario distribution rather than on operating time, so it is assessed with SOTIF and ISO/PAS 8800 methods.
</details>

## References

1. ISO 26262-9:2018. *Road vehicles — Functional safety — Part 9: Automotive safety integrity level (ASIL)-oriented and safety-oriented analyses* (clause 5 ASIL decomposition, clause 7 dependent failure analysis, clause 8 safety analyses). https://www.iso.org/standard/68391.html
2. ISO 26262-5:2018. *Road vehicles — Functional safety — Part 5: Product development at the hardware level* (hardware architectural metrics, PMHF). https://www.iso.org/standard/68387.html
3. IEC 61025:2006. *Fault tree analysis (FTA)*, 2nd edition. https://webstore.iec.ch/en/publication/4311
4. Vesely, W. E., Goldberg, F. F., Roberts, N. H., Haasl, D. F. (1981). *Fault Tree Handbook*, NUREG-0492. U.S. Nuclear Regulatory Commission. https://www.nrc.gov/reading-rm/doc-collections/nuregs/staff/sr0492/index.html
5. Fussell, J. B. (1975). How to hand-calculate system reliability and safety characteristics. *IEEE Transactions on Reliability*, R-24(3), 169–174. https://doi.org/10.1109/TR.1975.5215142
6. IEC 61508-6:2010. *Functional safety of E/E/PE safety-related systems — Part 6: Guidelines on the application of IEC 61508-2 and IEC 61508-3* (Annex D, beta-factor for common-cause failures). https://webstore.iec.ch/en/publication/5520
7. ISO 21448:2022. *Road vehicles — Safety of the intended functionality*. https://www.iso.org/standard/77490.html
8. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence*. https://www.iso.org/standard/83303.html

---
**Navigation:** ← [03 FMEA and FMEDA](03_fmea_and_fmeda.ipynb) · [Module index](README.md) · [05 Verification and validation](05_verification_and_validation.ipynb) →